In [ ]:
# ================= Sensitivity+PRCC+WinRates: FRC vs baselines on β0 × clustering ==============
# Metrics on holdout: RMSE, |Δt_peak| (in time), |ΔR(∞)| (final-size error).
# Baselines: Uniform (standard network SIR), EdgeBetweenness (advanced).
# Outputs:
#   • Heatmaps of Δ = (baseline error − FRC error) for each metric (positive ⇒ FRC better)
#   • PRCC (Partial Rank Correlation Coefficients) of Δ w.r.t β0 and clustering knob
#   • Aggregate win-rate table across families, saved as CSV.
# -----------------------------------------------------------------------------------------------

import os, math, random
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from typing import Dict, Tuple, List

# ------------------------------ Repro / config --------------------------------
SEED = 42
np.random.seed(SEED); random.seed(SEED)

OUTDIR = "sensitivity_plus_images"
os.makedirs(OUTDIR, exist_ok=True)

# Graph sizes tuned for speed/clarity
N = 600
P_ER_BASE = 0.004
WS_K = 10
BA_M_BASE = 3
PLC_M = 3
PLC_P_BASE = 0.05

# Time grid
T_MAX = 100.0
N_T   = 400
TIME  = np.linspace(0, T_MAX, N_T)
DT    = float(TIME[1] - TIME[0])

# SIR params
GAMMA = 0.1
ALPHA = 1.0           # beta_ij = beta0 * exp(ALPHA * z_std)
FIT_FRAC = 0.25
T_FIT_END = int(FIT_FRAC * N_T)
EPS = 1e-12

# Parameter grids
BETA_GRID = np.linspace(0.10, 0.60, 9)
WS_P_GRID = np.linspace(0.01, 0.30, 8)
PLC_P_GRID = np.linspace(0.01, 0.25, 8)
ER_P_GRID  = np.linspace(0.0025, 0.0075, 8)
BA_M_GRID  = [2,3,4,5,6,7,8,9]

# ===============================================================================================
#                                       Features
# ===============================================================================================

class FormanRicciUndirected:
    """Unit-weight undirected FRC; returns dict keyed by (min,max)."""
    def __init__(self, G: nx.Graph):
        if G.is_directed(): raise ValueError("Needs undirected graph")
        self.G = G
    def compute(self) -> Dict[Tuple, float]:
        F = {}
        for u, v in self.G.edges():
            we = wu = wv = 1.0
            su = sum(wu / math.sqrt(we * 1.0) for x in self.G.neighbors(u) if x != v)
            sv = sum(wv / math.sqrt(we * 1.0) for y in self.G.neighbors(v) if y != u)
            key = (u, v) if u < v else (v, u)
            F[key] = we * ((wu / we) + (wv / we) - su - sv)
        return F

def standardize(z: Dict[Tuple, float]) -> Dict[Tuple, float]:
    vals = np.array(list(z.values()), dtype=float)
    mu, sd = vals.mean(), vals.std()
    if not np.isfinite(sd) or sd <= 0: return {e: 0.0 for e in z}
    return {e: (v - mu) / sd for e, v in z.items()}

def feat_frc(G): return FormanRicciUndirected(G).compute()
def feat_uniform(G): return {tuple(sorted((u, v))): 0.0 for u, v in G.edges()}
def feat_edge_betweenness(G):
    eb = nx.edge_betweenness_centrality(G, normalized=True)
    return {tuple(sorted(k)): v for k, v in eb.items()}

# ===============================================================================================
#                                Build B & simulate (stable Euler)
# ===============================================================================================

def build_B(G: nx.Graph, z_raw: Dict[Tuple, float], beta0: float, alpha: float = 1.0):
    z = standardize(z_raw)
    nodes = list(G.nodes())
    idx = {v: i for i, v in enumerate(nodes)}
    n = len(nodes)
    B = np.zeros((n, n), dtype=np.float64)
    for u, v in G.edges():
        key = tuple(sorted((u, v)))
        w = beta0 * math.exp(alpha * z.get(key, 0.0))
        i, j = idx[u], idx[v]
        B[i, j] = w; B[j, i] = w
    np.fill_diagonal(B, 0.0)
    return B, nodes

def simulate_sir_euler(B: np.ndarray, seeds_idx: List[int], tvec: np.ndarray, gamma: float = GAMMA):
    n = B.shape[0]
    dt = float(tvec[1] - tvec[0])
    T = len(tvec)
    S = np.ones((T, n), dtype=np.float64)
    I = np.zeros((T, n), dtype=np.float64)
    R = np.zeros((T, n), dtype=np.float64)

    for i in seeds_idx:
        S[0, i] = 0.0
        I[0, i] = 1.0

    for t in range(T - 1):
        lam = B @ I[t]
        dS = -S[t] * lam
        dI = S[t] * lam - gamma * I[t]
        dR = gamma * I[t]
        S[t+1] = np.clip(S[t] + dt * dS, 0.0, 1.0)
        I[t+1] = np.clip(I[t] + dt * dI, 0.0, 1.0)
        R[t+1] = np.clip(R[t] + dt * dR, 0.0, 1.0)
        total = S[t+1] + I[t+1] + R[t+1] + EPS
        S[t+1] /= total; I[t+1] /= total; R[t+1] /= total
    return S, I, R

def pick_seeds_negfrc(G: nx.Graph, k: int, frc: Dict[Tuple, float]) -> List[int]:
    """Pick seeds among endpoints of negative-FRC edges, spaced by max–min distance."""
    if k <= 0: return []
    neg_nodes = set()
    for (u, v), f in frc.items():
        if f < 0:
            neg_nodes.add(u); neg_nodes.add(v)
    cand = list(neg_nodes) if neg_nodes else list(G.nodes())
    eb = feat_edge_betweenness(G)
    start = max(cand, key=lambda u: sum(eb.get(tuple(sorted((u, v))), 0.0) for v in G.neighbors(u)))
    chosen = [start]
    while len(chosen) < k:
        best_u, best_d = None, -1
        for u in cand:
            if u in chosen: continue
            d_min = np.inf
            for v in chosen:
                try: d = nx.shortest_path_length(G, u, v)
                except nx.NetworkXNoPath: d = np.inf
                d_min = min(d_min, d)
            d_min = 0 if not np.isfinite(d_min) else d_min
            if d_min > best_d:
                best_d, best_u = d_min, u
        if best_u is None:
            best_u = random.choice([x for x in G.nodes() if x not in chosen])
        chosen.append(best_u)
    return chosen

# ===============================================================================================
#                               Fitting & holdout metrics
# ===============================================================================================

def mse_fit_window(I_model, I_truth, t_end):
    e = I_model[:t_end] - I_truth[:t_end]
    return float(np.mean(e * e))

def golden_section_min(f, a, b, maxit=40, tol=1e-4):
    gr = (math.sqrt(5) - 1) / 2
    c = b - gr * (b - a)
    d = a + gr * (b - a)
    fc = f(c); fd = f(d)
    for _ in range(maxit):
        if abs(b - a) < tol: break
        if fc < fd:
            b, d, fd = d, c, fc
            c = b - gr * (b - a); fc = f(c)
        else:
            a, c, fc = c, d, fd
            d = a + gr * (b - a); fd = f(d)
    return 0.5 * (a + b)

def score_model(G, z_raw, seeds_idx, I_truth_mean, R_truth_mean, beta_bracket=(0.02, 1.2)):
    """Fit β0 (min fit-MSE), return beta_hat and holdout metrics."""
    def loss(beta):
        B, _ = build_B(G, z_raw, beta, ALPHA)
        S, I, R = simulate_sir_euler(B, seeds_idx, TIME, GAMMA)
        return mse_fit_window(I.mean(axis=1), I_truth_mean, T_FIT_END)

    beta_hat = golden_section_min(loss, beta_bracket[0], beta_bracket[1])
    # Score on holdout
    B, _ = build_B(G, z_raw, beta_hat, ALPHA)
    S, I, R = simulate_sir_euler(B, seeds_idx, TIME, GAMMA)
    I_mean = I.mean(axis=1)
    R_mean = R.mean(axis=1)

    rmse = float(np.sqrt(np.mean((I_mean[T_FIT_END:] - I_truth_mean[T_FIT_END:])**2)))
    tp_err = abs(int(np.argmax(I_mean)) - int(np.argmax(I_truth_mean))) * DT
    Rinf_err = abs(float(R_mean[-1]) - float(R_truth_mean[-1]))
    return beta_hat, rmse, tp_err, Rinf_err

def supercritical(B, gamma=GAMMA, s0=1.0):
    lam = float(np.max(np.linalg.eigvals(B).real))
    return (s0 * lam) > gamma

# ===============================================================================================
#                                      PRCC helpers
# ===============================================================================================

def _rank_dense(x: np.ndarray) -> np.ndarray:
    """Dense ranks (ties share same rank order)."""
    order = np.argsort(x)
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(x) + 1, dtype=float)
    return ranks

def prcc_two_inputs(delta: np.ndarray, beta: np.ndarray, p: np.ndarray):
    """PRCC of Δ (response) with β0 and p controlling for the other input."""
    # rank transform
    Y = _rank_dense(delta)
    X1 = _rank_dense(beta)
    X2 = _rank_dense(p)

    # regress residuals: rY = Y ~ X2 ; rX1 = X1 ~ X2 ; corr(rY, rX1)
    def residual(y, x):  # simple OLS residuals for a single regressor
        x1 = np.vstack([x, np.ones_like(x)]).T
        coef, *_ = np.linalg.lstsq(x1, y, rcond=None)
        return y - x1 @ coef

    rY_X2  = residual(Y, X2)
    rX1_X2 = residual(X1, X2)
    rY_X1  = residual(Y, X1)
    rX2_X1 = residual(X2, X1)

    def corr(a, b):
        a = a - a.mean(); b = b - b.mean()
        den = np.sqrt((a*a).sum() * (b*b).sum())
        return 0.0 if den == 0 else float((a*b).sum() / den)

    prcc_beta = corr(rY_X2, rX1_X2)  # Δ ~ β | p
    prcc_p    = corr(rY_X1, rX2_X1)  # Δ ~ p | β
    return prcc_beta, prcc_p

# ===============================================================================================
#                          Sensitivity for one topology (heatmaps + PRCC)
# ===============================================================================================

def sensitivity_family(family: str):
    """
    Build heatmaps of Δ (baseline − FRC) for three metrics (RMSE, tpeak, Rinf),
    compute PRCC of Δ with β0 and p (or m), and return win-rate stats.
    """
    # family setup
    if family == "WS":
        p_list = WS_P_GRID
        def makeG(p, seed): return nx.watts_strogatz_graph(N, WS_K, float(p), seed=seed)
        p_label = "WS rewiring p"
    elif family == "PLC":
        p_list = PLC_P_GRID
        def makeG(p, seed): return nx.powerlaw_cluster_graph(N, PLC_M, float(p), seed=seed)
        p_label = "PLC triangle p"
    elif family == "ER":
        p_list = ER_P_GRID
        def makeG(p, seed): return nx.erdos_renyi_graph(N, float(p), seed=seed)
        p_label = "ER edge prob"
    elif family == "BA":
        p_list = BA_M_GRID
        def makeG(p, seed): return nx.barabasi_albert_graph(N, int(p), seed=seed)
        p_label = "BA attachment m"
    else:
        raise ValueError("family must be one of {WS, PLC, ER, BA}")

    betas = BETA_GRID

    # matrices for Δ vs Uniform and Δ vs EB (for each metric)
    shape = (len(p_list), len(betas))
    def mats(): return (np.full(shape, np.nan), np.full(shape, np.nan))
    d_rmse_U, d_rmse_EB = mats()
    d_tpk_U,  d_tpk_EB  = mats()
    d_rinf_U, d_rinf_EB = mats()

    # PRCC accumulators (flattened valid cells)
    beta_vec = []
    p_vec    = []
    vec_rmse_U = []; vec_rmse_EB = []
    vec_tpk_U  = []; vec_tpk_EB  = []
    vec_rinf_U = []; vec_rinf_EB = []

    for i, pv in enumerate(p_list):
        G = makeG(pv, seed=SEED + i)
        z_truth = feat_frc(G)
        EB_feat = feat_edge_betweenness(G)  # compute once per graph
        U_feat  = feat_uniform(G)

        for j, btrue in enumerate(betas):
            # truth
            B_true, nodes = build_B(G, z_truth, btrue, ALPHA)
            if not supercritical(B_true, GAMMA):
                continue
            seeds_nodes = pick_seeds_negfrc(G, k=6, frc=z_truth)
            idx = {v: k for k, v in enumerate(nodes)}
            seeds_idx = [idx[u] for u in seeds_nodes]
            S_t, I_t, R_t = simulate_sir_euler(B_true, seeds_idx, TIME, GAMMA)
            I_truth_mean = I_t.mean(axis=1)
            R_truth_mean = R_t.mean(axis=1)

            # FRC model (fit β0)
            beta_frc, rmse_frc, tpk_frc, rinf_frc = score_model(G, z_truth, seeds_idx, I_truth_mean, R_truth_mean)
            # Uniform
            beta_uni, rmse_uni, tpk_uni, rinf_uni = score_model(G, U_feat, seeds_idx, I_truth_mean, R_truth_mean)
            # EdgeBetweenness
            beta_eb,  rmse_eb,  tpk_eb,  rinf_eb  = score_model(G, EB_feat, seeds_idx, I_truth_mean, R_truth_mean)

            d_rmse_U[i, j] = rmse_uni - rmse_frc
            d_rmse_EB[i, j]= rmse_eb  - rmse_frc
            d_tpk_U[i, j]  = tpk_uni  - tpk_frc
            d_tpk_EB[i, j] = tpk_eb   - tpk_frc
            d_rinf_U[i, j] = rinf_uni - rinf_frc
            d_rinf_EB[i, j]= rinf_eb  - rinf_frc

            beta_vec.append(btrue)
            p_vec.append(pv)
            vec_rmse_U.append(d_rmse_U[i, j]); vec_rmse_EB.append(d_rmse_EB[i, j])
            vec_tpk_U.append(d_tpk_U[i, j]);   vec_tpk_EB.append(d_tpk_EB[i, j])
            vec_rinf_U.append(d_rinf_U[i, j]); vec_rinf_EB.append(d_rinf_EB[i, j])

    def heatmap(Z, title, p_vals, b_vals, ylabel, fname):
        plt.figure(figsize=(8.4, 5.0), dpi=120)
        # Use symmetric color limits around 0 for interpretability
        vmax = np.nanmax(np.abs(Z)); vmin = -vmax
        im = plt.imshow(Z, aspect="auto", origin="lower",
                        extent=[b_vals[0], b_vals[-1], 0, len(p_vals)-1],
                        cmap="RdYlGn", vmin=vmin, vmax=vmax)
        ylocs = np.arange(len(p_vals))
        plt.yticks(ylocs, [str(v) for v in p_vals])
        plt.xlabel("True β₀ (hidden truth)"); plt.ylabel(ylabel)
        plt.title(title)
        cbar = plt.colorbar(im)
        cbar.set_label("Δ = error(baseline) − error(FRC)  (positive ⇒ FRC better)")
        plt.tight_layout()
        plt.savefig(os.path.join(OUTDIR, fname), dpi=300)
        plt.close()

    # Heatmaps: RMSE, tpeak, Rinf (Uniform and EB)
    heatmap(d_rmse_U, f"{family}: ΔRMSE (Uniform − FRC)", p_list, betas, p_label,
            f"{family}_dRMSE_uniform_vs_FRC.png")
    heatmap(d_rmse_EB, f"{family}: ΔRMSE (EdgeBetweenness − FRC)", p_list, betas, p_label,
            f"{family}_dRMSE_eb_vs_FRC.png")

    heatmap(d_tpk_U, f"{family}: Δ|t_peak| (Uniform − FRC)", p_list, betas, p_label,
            f"{family}_dTpeak_uniform_vs_FRC.png")
    heatmap(d_tpk_EB, f"{family}: Δ|t_peak| (EdgeBetweenness − FRC)", p_list, betas, p_label,
            f"{family}_dTpeak_eb_vs_FRC.png")

    heatmap(d_rinf_U, f"{family}: Δ|R(∞)| (Uniform − FRC)", p_list, betas, p_label,
            f"{family}_dRinf_uniform_vs_FRC.png")
    heatmap(d_rinf_EB, f"{family}: Δ|R(∞)| (EdgeBetweenness − FRC)", p_list, betas, p_label,
            f"{family}_dRinf_eb_vs_FRC.png")

    # Win-rates (% cells where FRC better)
    def winrate(Z):
        mask = ~np.isnan(Z)
        return float(np.sum((Z > 0) & mask)) / float(np.sum(mask)) if np.sum(mask) else np.nan

    wins = {
        "RMSE_vsUniform": winrate(d_rmse_U),
        "RMSE_vsEB":      winrate(d_rmse_EB),
        "Tpeak_vsUniform":winrate(d_tpk_U),
        "Tpeak_vsEB":     winrate(d_tpk_EB),
        "Rinf_vsUniform": winrate(d_rinf_U),
        "Rinf_vsEB":      winrate(d_rinf_EB),
    }

    # PRCC for ΔRMSE (most central metric); you can uncomment to add for the others.
    beta_vec = np.array(beta_vec, dtype=float)
    p_vec    = np.array(p_vec, dtype=float)
    vec_rmse_U = np.array(vec_rmse_U, dtype=float)
    vec_rmse_EB= np.array(vec_rmse_EB, dtype=float)

    prcc_U = prcc_two_inputs(vec_rmse_U, beta_vec, p_vec)
    prcc_EB= prcc_two_inputs(vec_rmse_EB, beta_vec, p_vec)

    print(f"[{family}] Win-rate (ΔRMSE>0) FRC better than Uniform: {100*wins['RMSE_vsUniform']:.1f}% ; "
          f"than EdgeBetweenness: {100*wins['RMSE_vsEB']:.1f}%")
    print(f"[{family}] PRCC of ΔRMSE with β0|p: {prcc_U[0]:+.3f} ; with p|β0: {prcc_U[1]:+.3f} (Uniform)")
    print(f"[{family}] PRCC of ΔRMSE with β0|p: {prcc_EB[0]:+.3f} ; with p|β0: {prcc_EB[1]:+.3f} (EdgeBetw.)")

    return {
        "family": family,
        "betas": betas, "p_list": p_list,
        "d_rmse_U": d_rmse_U, "d_rmse_EB": d_rmse_EB,
        "d_tpk_U": d_tpk_U,   "d_tpk_EB": d_tpk_EB,
        "d_rinf_U": d_rinf_U, "d_rinf_EB": d_rinf_EB,
        "wins": wins,
        "prcc_rmse_uniform": prcc_U,
        "prcc_rmse_eb": prcc_EB
    }

# ===============================================================================================
#                                              Run
# ===============================================================================================

def main():
    all_stats = []
    for fam in ["WS", "PLC", "ER", "BA"]:
        print(f"\n=== Sensitivity+PRCC: {fam} ===")
        stats = sensitivity_family(fam)
        all_stats.append(stats)

    # Aggregate win-rates into a CSV-like table
    headers = ["Family",
               "Win ΔRMSE vs Uniform",
               "Win ΔRMSE vs EdgeBetw",
               "Win Δ|tpeak| vs Uniform",
               "Win Δ|tpeak| vs EdgeBetw",
               "Win Δ|R(∞)| vs Uniform",
               "Win Δ|R(∞)| vs EdgeBetw",
               "PRCC(ΔRMSE,β0|p)[Uniform]",
               "PRCC(ΔRMSE,p|β0)[Uniform]",
               "PRCC(ΔRMSE,β0|p)[EdgeBetw]",
               "PRCC(ΔRMSE,p|β0)[EdgeBetw]"]
    lines = [",".join(headers)]
    print("\n=== Aggregate win-rate & PRCC summary ===")
    for s in all_stats:
        w = s["wins"]
        prU = s["prcc_rmse_uniform"]
        prE = s["prcc_rmse_eb"]
        row = [
            s["family"],
            f"{100*w['RMSE_vsUniform']:.1f}%",
            f"{100*w['RMSE_vsEB']:.1f}%",
            f"{100*w['Tpeak_vsUniform']:.1f}%",
            f"{100*w['Tpeak_vsEB']:.1f}%",
            f"{100*w['Rinf_vsUniform']:.1f}%",
            f"{100*w['Rinf_vsEB']:.1f}%",
            f"{prU[0]:+.3f}", f"{prU[1]:+.3f}",
            f"{prE[0]:+.3f}", f"{prE[1]:+.3f}",
        ]
        print("{:<4s}  ".format(row[0]) +
              "  ".join(f"{h}: {v}" for h, v in zip(headers[1:], row[1:])))
        lines.append(",".join(row))

    csv_path = os.path.join(OUTDIR, "sensitivity_summary.csv")
    with open(csv_path, "w") as f:
        f.write("\n".join(lines))
    print(f"\nSaved heatmaps + summary CSV in: {OUTDIR}\n{csv_path}")

if __name__ == "__main__":
    main()



=== Sensitivity+PRCC: WS ===
[WS] Win-rate (ΔRMSE>0) FRC better than Uniform: 100.0% ; than EdgeBetweenness: 100.0%
[WS] PRCC of ΔRMSE with β0|p: -0.846 ; with p|β0: -0.429 (Uniform)
[WS] PRCC of ΔRMSE with β0|p: -0.930 ; with p|β0: -0.713 (EdgeBetw.)

=== Sensitivity+PRCC: PLC ===
[PLC] Win-rate (ΔRMSE>0) FRC better than Uniform: 100.0% ; than EdgeBetweenness: 100.0%
[PLC] PRCC of ΔRMSE with β0|p: -0.991 ; with p|β0: +0.563 (Uniform)
[PLC] PRCC of ΔRMSE with β0|p: -0.993 ; with p|β0: +0.573 (EdgeBetw.)

=== Sensitivity+PRCC: ER ===
[ER] Win-rate (ΔRMSE>0) FRC better than Uniform: 100.0% ; than EdgeBetweenness: 100.0%
[ER] PRCC of ΔRMSE with β0|p: -0.794 ; with p|β0: -0.862 (Uniform)
[ER] PRCC of ΔRMSE with β0|p: -0.771 ; with p|β0: -0.692 (EdgeBetw.)

=== Sensitivity+PRCC: BA ===
[BA] Win-rate (ΔRMSE>0) FRC better than Uniform: 100.0% ; than EdgeBetweenness: 100.0%
[BA] PRCC of ΔRMSE with β0|p: -0.801 ; with p|β0: -0.803 (Uniform)
[BA] PRCC of ΔRMSE with β0|p: -0.925 ; with p|β0: -0.